## Problem Statement : Hospital Patient Data Analysis
## Context:
A hospital maintains patient records including admission details, department, diagnosis, doctor, and bill amount. You have two datasets: one with patient info and another with billing details. Some patients have blank bill amounts, and there are multiple rows for the same patient due to follow-ups.

## Tasks:

## Importing libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as norm

## 1.Load the patient dataset and show summary with info().

In [66]:
patient_df = pd.read_csv('Patient_Data.csv')
patient_df.head()

,PatientID,Name,Department,Doctor,BillAmount,ReceptionistID,CheckInTime
0,101,Alice,Cardiology,Dr. Smith,5000.0,1,2023-01-10 09:00
1,102,Bob,Neurology,Dr. John,NaN,2,2023-01-11 10:30
2,103,Charlie,Orthopedics,Dr. Lee,7500.0,1,2023-01-12 11:00
3,104,David,Cardiology,Dr. Smith,6200.0,3,2023-01-13 12:00
4,105,Eva,Dermatology,Dr. Rose,NaN,2,2023-01-14 08:45


In [67]:
#showing the sumary with information
patient_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes


In [68]:
patient_df.isnull().sum()

PatientID         0
Name              0
Department        0
Doctor            0
BillAmount        2
ReceptionistID    0
CheckInTime       0
dtype: int64

## 2.Select only the columns relevant for billing: ['PatientID', 'Department', 'Doctor', 'BillAmount'].

In [69]:
x = patient_df[['PatientID','Department','Doctor','BillAmount']]
print(x)

   PatientID   Department     Doctor  BillAmount
0        101   Cardiology  Dr. Smith      5000.0
1        102    Neurology   Dr. John         NaN
2        103  Orthopedics    Dr. Lee      7500.0
3        104   Cardiology  Dr. Smith      6200.0
4        105  Dermatology   Dr. Rose         NaN
5        101   Cardiology  Dr. Smith      5000.0


## 3.Drop administrative columns like ['ReceptionistID', 'CheckInTime'].

In [70]:
x = patient_df.drop(['ReceptionistID', 'CheckInTime'],axis = 1)
print(x)

   PatientID     Name   Department     Doctor  BillAmount
0        101    Alice   Cardiology  Dr. Smith      5000.0
1        102      Bob    Neurology   Dr. John         NaN
2        103  Charlie  Orthopedics    Dr. Lee      7500.0
3        104    David   Cardiology  Dr. Smith      6200.0
4        105      Eva  Dermatology   Dr. Rose         NaN
5        101    Alice   Cardiology  Dr. Smith      5000.0


## 4.Use groupby to find total bill amount per department.

In [71]:
x = patient_df.groupby('Department')['BillAmount'].sum()
print(x)

Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


## 5.Remove duplicate patient records based on PatientID.

In [72]:
#checking the duplicate columns
patient_df.duplicated()

0    False
1    False
2    False
3    False
4    False
5     True
dtype: bool

In [88]:
#removing the duplicate values
x = patient_df.drop_duplicates(subset = 'PatientID')  #drop_duplicate --> removes the repeated values #subset-->it represents uniqueness
print(x)

   PatientID     Name   Department     Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology  Dr. Smith      5000.0               1   
1        102      Bob    Neurology   Dr. John      5925.0               2   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0               1   
3        104    David   Cardiology  Dr. Smith      6200.0               3   
4        105      Eva  Dermatology   Dr. Rose      5925.0               2   

        CheckInTime  
0  2023-01-10 09:00  
1  2023-01-11 10:30  
2  2023-01-12 11:00  
3  2023-01-13 12:00  
4  2023-01-14 08:45  


## 6.Fill missing BillAmount values with the mean bill amount.

In [93]:
x = patient_df.fillna(df['BillAmount'].mean(),inplace = True)           #fillna --> it fills the missing values based on given inplace value
print(x)

None


## 7.Merge the billing dataset with patient dataset on PatientID.

In [77]:
# read the billing dataset
billing_df = pd.read_csv('Billing_Data.csv')
print(billing_df)

   PatientID  InsuranceCovered  FinalAmount
0        101              2000         3000
1        102              1500         3500
2        103              2500         5000
3        104              3000         3200
4        105              1000         4000


In [92]:
merged_df = pd.merge(billing_df,patient_df,on='PatientID',how = 'inner')  #merge -->it combines the data
print(merged_df)

   PatientID  InsuranceCovered  FinalAmount     Name   Department     Doctor  \
0        101              2000         3000    Alice   Cardiology  Dr. Smith   
1        101              2000         3000    Alice   Cardiology  Dr. Smith   
2        102              1500         3500      Bob    Neurology   Dr. John   
3        103              2500         5000  Charlie  Orthopedics    Dr. Lee   
4        104              3000         3200    David   Cardiology  Dr. Smith   
5        105              1000         4000      Eva  Dermatology   Dr. Rose   

   BillAmount  ReceptionistID       CheckInTime  
0      5000.0               1  2023-01-10 09:00  
1      5000.0               1  2023-01-10 09:00  
2      5925.0               2  2023-01-11 10:30  
3      7500.0               1  2023-01-12 11:00  
4      6200.0               3  2023-01-13 12:00  
5      5925.0               2  2023-01-14 08:45  


## 8.Concatenate an additional DataFrame that contains new patients for the current week (row-wise).

In [82]:
#creating new patient data
new_patients_df = pd.DataFrame({
    'PatientID': [101, 102],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr. Sharma', 'Dr. Rao'],
    'BillAmount': [5000, 7000]
})
print(new_patients_df)

   PatientID  Department      Doctor  BillAmount
0        101  Cardiology  Dr. Sharma        5000
1        102   Neurology     Dr. Rao        7000


In [91]:
#concatenate new patient data with old patient data
final_patient_df = pd.concat([patient_df,new_patients_df],axis = 0,ignore_index = True) #concat --> it adds two rows or columns into one column or row
print(final_patient_df)

   PatientID     Name   Department      Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology   Dr. Smith      5000.0             1.0   
1        102      Bob    Neurology    Dr. John      5925.0             2.0   
2        103  Charlie  Orthopedics     Dr. Lee      7500.0             1.0   
3        104    David   Cardiology   Dr. Smith      6200.0             3.0   
4        105      Eva  Dermatology    Dr. Rose      5925.0             2.0   
5        101    Alice   Cardiology   Dr. Smith      5000.0             1.0   
6        101      NaN   Cardiology  Dr. Sharma      5000.0             NaN   
7        102      NaN    Neurology     Dr. Rao      7000.0             NaN   

        CheckInTime  
0  2023-01-10 09:00  
1  2023-01-11 10:30  
2  2023-01-12 11:00  
3  2023-01-13 12:00  
4  2023-01-14 08:45  
5  2023-01-10 09:00  
6               NaN  
7               NaN  


## 9.Concatenate new billing category columns like ['InsuranceCovered', 'FinalAmount'] (column-wise).

In [90]:
#create biling extra dataset
billing_extra_df = pd.DataFrame({
    'InsuranceCovered': ['Yes', 'No', 'Yes'],
    'FinalAmount': [5000, 8000, 3000]
})
#Add the new billing extra data
complete_df = pd.concat([patient_df,
                         billing_extra_df],
                        axis=1)           #axis = 1 represents 'columns' and 0 represents 'rows'

print(complete_df.head())

   PatientID     Name   Department     Doctor  BillAmount  ReceptionistID  \
0        101    Alice   Cardiology  Dr. Smith      5000.0               1   
1        102      Bob    Neurology   Dr. John      5925.0               2   
2        103  Charlie  Orthopedics    Dr. Lee      7500.0               1   
3        104    David   Cardiology  Dr. Smith      6200.0               3   
4        105      Eva  Dermatology   Dr. Rose      5925.0               2   

        CheckInTime InsuranceCovered  FinalAmount  
0  2023-01-10 09:00              Yes       5000.0  
1  2023-01-11 10:30               No       8000.0  
2  2023-01-12 11:00              Yes       3000.0  
3  2023-01-13 12:00              NaN          NaN  
4  2023-01-14 08:45              NaN          NaN  


## Conclusion

After performing these preprocessing steps:

Missing bill values are handled.
Duplicate patient records are removed.
Patient and billing datasets are integrated.
Department-wise revenue analysis becomes possible.
The final dataset is clean and ready for advanced analytics or machine learning.